In [102]:
import gensim
import pickle as pk
import pandas as pd

In [103]:
with open('neg_sent.pkl','rb') as file:
    neg_sent = pk.load(file)

In [104]:
comments = neg_sent['COMMENT']
corpus = [comment for comment in comments]

In [105]:
stoplist = set('the and is in to it you that he was for on are as with his they at be this have from or one had by word but not what all were we when your can said there use an each which she do how their if 1 2 3 4 5 6 7 8 9 i a game of play my very great so more played its fun de la like just best ever love cards deck amazing really some monster marvel coop games first playing experience players after much than crew un horror arkham will me also dungeon top get nemesis expansion good out pandemic legacy ive weve gloomhaven original would scenarios app up lot has que en y et le jeu only now through still im far again favorite awesome perfect spirit á des el 14 - az about vs les dont been into because too player plays bit feel characters other them 10 edition win mage knight fantastic island ah robinson crusoe 50 outstanding excellent á favorites plus better am new different well many make way pack eh death – — con where two spirits lcg our every got cant one three four five six seven eight nine ten no over update rating rich perfectly expansions even need people could though interesting monsters season group us chapters madness kickstarter lion frosthaven jaws mansions cthulhu masterpiece 1st heros ancient alien never times most always trick taking sleeved 2020 cursed 2021 forsaken down want then once favourite who wife wait juego earth es se est dd things few while going did see second forward feels superb backed 2022 endless black hogy king du une promo il lots didntbefore back enjoyed worth set 7th deckbuilder continent go end next little difficult quite highly 24 it\'s i\'ve i\'m don\'t 2019 before lo los para las una o mut pero w und das dunwich mythos investigator der ist zu spiel ich ein nicht mit carcosa aber den auf auch sehr eine werden te por si al als sin je como del 2017 sich sind von kann noch'.split(' '))
cleaned_corpus = [[word for word in doc.lower().split() if word not in stoplist]
                  for doc in corpus]

In [106]:
from collections import defaultdict
import pprint

frequency = defaultdict(int)
for texts in cleaned_corpus:
    for token in texts:
        frequency[token] += 1

processed_corpus = [[token for token in text if frequency[token] > 1]
                    for text in cleaned_corpus]

In [107]:
from gensim import corpora

dictionary = corpora.Dictionary(processed_corpus)

In [108]:
bow_corpus = [dictionary.doc2bow(text) for text in processed_corpus]

In [109]:
from gensim import models

tfidf = models.TfidfModel(bow_corpus)

In [110]:
words = "good good mechanics".split()

In [111]:
with open('neg_sent_tfidf_model.pkl','wb') as file:
    pk.dump(tfidf, file)

In [112]:
important_words = []

for doc in corpus:
    result = tfidf[dictionary.doc2bow(doc.lower().split())]
    for weight in result:
        if weight[1] > 0.70:
            important_words.append(dictionary[weight[0]])

In [113]:
len(important_words)

1045

In [114]:
from gensim.models.ldamodel import LdaModel

lda_model = LdaModel(bow_corpus, num_topics=3, id2word=dictionary, passes=15)

topics = lda_model.print_topics(num_words=10)
for topic in topics:
    print(topic)

(0, '0.009*"return" + 0.008*"forgotten" + 0.007*"cycle" + 0.006*"age" + 0.006*"muy" + 0.006*"card" + 0.005*"path" + 0.005*"más" + 0.005*"token" + 0.004*"campaign"')
(1, '0.009*"time" + 0.006*"story" + 0.006*"rules" + 0.004*"card" + 0.004*"hard" + 0.004*"bad" + 0.004*"board" + 0.004*"scenario" + 0.003*"campaign" + 0.003*"think"')
(2, '0.044*"die" + 0.009*"man" + 0.006*"für" + 0.006*"eldritch" + 0.006*"lore" + 0.005*"remnants" + 0.005*"strange" + 0.005*"ruin" + 0.005*"hat" + 0.004*"gut"')


Per Chat GPT:
- Topic 1: Frustration with Campaign or Theme Repetition
- Topic 2: Critique of Story and Rule Complexity
- Topic 3: Disappointment with Dark Themes or Overused Lore